# qec-bench — Kaggle training run

Trains the per-distance neural decoders on Kaggle's free GPU tier. Enable
**Settings -> Accelerator -> GPU** and **Settings -> Internet -> On**.

Every shell step runs through `sh()`, which raises on a non-zero exit (so a
failure surfaces as an ERROR status, not a silently 'complete' run) and tees
output to `/kaggle/working/run.log`. Only the checkpoints and that log are
written to `/kaggle/working` — the dataset goes to ephemeral `/tmp` — so the
downloadable output stays small.

In [ ]:
# Config for this run (see configs/ in the repo).
DATASET = "train_v2"
TRAIN_CONFIG = "mlp_v3"

import os
import subprocess
import sys

WORK = "/kaggle/working"
DATA = "/tmp/qec-data"
WEIGHTS = os.path.join(WORK, "weights")
LOG = os.path.join(WORK, "run.log")
os.makedirs(WEIGHTS, exist_ok=True)
open(LOG, "w").close()  # fresh log each run


def sh(cmd):
    """Run a shell command, tee output to run.log, raise on non-zero exit."""
    print(">>>", cmd, flush=True)
    with open(LOG, "a") as lf:
        print(">>>", cmd, file=lf, flush=True)
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            lf.write(line)
        p.wait()
    if p.returncode != 0:
        raise RuntimeError("FAILED (%d): %s" % (p.returncode, cmd))


In [ ]:
sh("nvidia-smi -L")

In [ ]:
# Clone the repo and install qecbench. Kaggle already ships numpy + torch,
# so install only the deps its base image lacks, then the package itself
# with --no-deps to avoid disturbing the preinstalled torch.
if not os.path.isdir("/kaggle/working/qec-bench"):
    sh("git clone --depth 1 https://github.com/Lucas-Maingi/qec-bench.git /kaggle/working/qec-bench")
os.chdir("/kaggle/working/qec-bench")
sh("pip install -q stim pymatching pyyaml")
sh("pip install -q -e . --no-deps")
sh("python -c \"import qecbench, torch; print('qecbench', qecbench.__version__, 'torch', torch.__version__, 'cuda', torch.cuda.is_available())\"")


## Select a working training device

In [ ]:
# Pick the training device. torch.cuda.is_available() is not enough: Kaggle
# sometimes assigns a Tesla P100 (sm_60), which the preinstalled PyTorch build
# (cu128, sm_70+) cannot actually run kernels on. Run a real GPU compute op; if
# it fails, install a P100-compatible torch and re-test; fall back to CPU only
# if a working GPU truly can't be had.
CUDA_PROBE = "import torch; (torch.zeros(8, device='cuda') + 1).sum().item(); print('cuda-ok')"


def cuda_usable():
    r = subprocess.run(["python", "-c", CUDA_PROBE], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[-300:])
    return r.returncode == 0


if cuda_usable():
    DEVICE = "cuda"
else:
    print("default torch cannot run on this GPU; installing a compatible build")
    # --no-deps: torch's deps are already present from Kaggle's build; the
    # cu121 index doesn't host all of them, so resolving them there would fail.
    sh("pip install -q torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121 --no-deps")
    DEVICE = "cuda" if cuda_usable() else "cpu"
print("training device:", DEVICE)


## 1/2 — Training dataset (generated to /tmp, ~10 min)

In [ ]:
sh("qecbench generate --config configs/datasets/%s.yaml --out %s" % (DATASET, DATA))

## 2/2 — Train the per-distance decoders

In [ ]:
sh("qecbench train --config configs/train/%s.yaml --data %s/%s --out %s --device %s"
   % (TRAIN_CONFIG, DATA, DATASET, WEIGHTS, DEVICE))

## Result — checkpoints land in /kaggle/working/weights

Commit the notebook (*Save & Run All*) so the output persists, then download
the files under `weights/` from the Output tab (a few MB).

In [ ]:
for f in sorted(os.listdir(WEIGHTS)):
    print(f, os.path.getsize(os.path.join(WEIGHTS, f)) // 1024, "KB")
print("--- tail of run.log ---")
print("".join(open(LOG).readlines()[-15:]))
